
# 22B — V5 DEV Expansion E2 Roster Freeze

This notebook runs **after** the combined Original DEV + E1 necessary-condition gate
has failed.

It does **not** inspect events, pair gaps, chronology, astrology, Control, or CONFIRM events.

E2 is intentionally another **whole** wave:

- COMPETITIVE 40
- PROJECT 50
- STATUS 70
- TOTAL 160

Even though the observed remaining deficit is PROJECT-only, PROJECT is **not**
oversampled. This preserves the previously declared whole-wave expansion design.

If fresh exact-DOB candidate supply is insufficient after excluding Original DEV,
sealed CONFIRM, and E1, this notebook stops and writes a supply diagnostic.
**Do not lower quotas.**


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_VERSION = "SAJU_ML_V5_DEV_EXPANSION_E2_ROSTER_FREEZE_20260817"
SEED = 2026081704
AXES = ["COMPETITIVE", "PROJECT", "STATUS"]
QUOTAS = {"COMPETITIVE": 40, "PROJECT": 50, "STATUS": 70}

def repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p] + list(p.parents):
        if (c / "saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repo.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def norm_name(x):
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.casefold()
    return re.sub(r"[^a-z0-9]+", "", s)

def selection_key(axis, qid, name):
    raw = f"{SEED}|{axis}|{qid}|{norm_name(name)}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

ROOT = repo_root()

CORPUS = ROOT / "research/ml_corpus/v5_ground_truth"
COMBINED = ROOT / "research/ml/artifacts/v5_combined_prepair_feasibility"
REPAIR = ROOT / "research/ml/artifacts/v5_identity_repair"
AUDIT = ROOT / "research/ml/artifacts/v5_identity_linkage_audit"
E1 = ROOT / "research/ml/artifacts/v5_dev_expansion_e1"

OUT = ROOT / "research/ml/artifacts/v5_dev_expansion_e2"
BATCH_OUT = OUT / "batches"
OUT.mkdir(parents=True, exist_ok=True)
BATCH_OUT.mkdir(parents=True, exist_ok=True)

PROTO = CORPUS / "V5_DEV_EXPANSION_WAVE_E2_PROTOCOL.json"
COMBINED_DEC = COMBINED / "V5_COMBINED_PREPAIR_FEASIBILITY_DECISION.json"
DEV = REPAIR / "V5_DEV_SUBJECT_ROSTER_160_REPAIRED.csv"
CONF = REPAIR / "V5_CONFIRM_SUBJECT_ROSTER_80_REPAIRED_SEALED.csv"
POOL = AUDIT / "V5_IDENTITY_LINKAGE_AUDIT_CANDIDATE_POOL.csv"
E1_ROSTER = E1 / "V5_DEV_EXPANSION_E1_ROSTER_160.csv"
E1_DEC = E1 / "V5_DEV_EXPANSION_E1_FREEZE_DECISION.json"
E1_STATUS_POOL = E1 / "V5_DEV_EXPANSION_E1_STATUS_SUPPLEMENT_EXACT_DOB_POOL.csv"

required = [PROTO, COMBINED_DEC, DEV, CONF, POOL, E1_ROSTER, E1_DEC]
for p in required:
    if not p.exists():
        raise FileNotFoundError(p)

proto = json.load(open(PROTO, encoding="utf-8"))
cdec = json.load(open(COMBINED_DEC, encoding="utf-8"))
e1dec = json.load(open(E1_DEC, encoding="utf-8"))

assert proto["status"] == (
    "PREDECLARED_AFTER_COMBINED_PREPAIR_FAILURE_BEFORE_E2_MEMBERSHIP_SELECTION"
)
assert cdec["status"] == "V5_COMBINED_PREPAIR_FEASIBILITY_FAIL_EXPANSION_E2_REQUIRED"
assert cdec["pair_gap_inspected"] is False
assert cdec["pairs_generated"] is False
assert cdec["astrology_generated"] is False
assert cdec["control_scored"] is False
assert cdec["confirm_researched"] is False
assert cdec["thresholds_changed_after_observation"] is False
assert cdec["project_oversampling_allowed_for_E2"] is False
assert cdec["E2_axis_quotas_if_fail"] == QUOTAS

assert e1dec["status"] == "V5_DEV_EXPANSION_E1_ROSTER_FROZEN_READY_FOR_EVENT_COLLECTION"
assert e1dec["n_subjects"] == 160
assert e1dec["axis_counts"] == QUOTAS

dev = pd.read_csv(DEV)
conf = pd.read_csv(CONF)
e1 = pd.read_csv(E1_ROSTER)
pool = pd.read_csv(POOL)

assert len(dev) == 160
assert len(conf) == 80
assert len(e1) == 160
assert sha256_file(E1_ROSTER) == e1dec["roster_sha256"]

print("22B PREFLIGHT PASS")
print("Combined gate:", cdec["status"])
print("No event corpus or pair file loaded.")


22B PREFLIGHT PASS
Combined gate: V5_COMBINED_PREPAIR_FEASIBILITY_FAIL_EXPANSION_E2_REQUIRED
No event corpus or pair file loaded.


## 1. Build fresh exact-DOB candidate supply without outcomes

In [2]:

# Base independently audited exact-DOB candidate pool.
cand = pool[
    (pool["linkage_status"] == "PASS_EXACT_DOB")
    & pool["wikidata_id"].notna()
].copy()

if "axis" not in cand.columns:
    raise RuntimeError("Candidate pool missing frozen axis column.")

cand["norm_name_e2"] = cand["name"].map(norm_name)
cand["_source_priority"] = 0
cand["_candidate_source_e2"] = cand.get(
    "candidate_source",
    pd.Series(["V5_IDENTITY_LINKAGE_AUDIT_CANDIDATE_POOL"] * len(cand),
              index=cand.index)
)

# Reuse the already frozen E1 STATUS *candidate source* if it exists.
# This is not event data. It was created outcome-blind using role + exact-DOB linkage.
if E1_STATUS_POOL.exists():
    supp = pd.read_csv(E1_STATUS_POOL).copy()
    if len(supp):
        assert (supp["linkage_status"] == "PASS_EXACT_DOB").all()
        supp["norm_name_e2"] = supp["name"].map(norm_name)
        supp["_source_priority"] = 1
        supp["_candidate_source_e2"] = (
            "V5_E1_STATUS_ROLE_SUPPLEMENT_EXACT_DOB"
        )

        # Schema-align without changing values.
        for c in cand.columns:
            if c not in supp.columns:
                supp[c] = np.nan
        for c in supp.columns:
            if c not in cand.columns:
                cand[c] = np.nan

        cand = pd.concat(
            [cand[supp.columns], supp],
            ignore_index=True,
            sort=False
        )

# Exclude ALL prior protected identities: repaired DEV, sealed CONFIRM, and E1.
protected_frames = [dev, conf, e1]
protected_names = set()
protected_qids = set()

for frame in protected_frames:
    protected_names |= set(frame["name"].map(norm_name))
    if "wikidata_id" in frame.columns:
        protected_qids |= set(
            frame["wikidata_id"].dropna().astype(str)
        )

cand = cand[
    ~cand["norm_name_e2"].isin(protected_names)
    & ~cand["wikidata_id"].astype(str).isin(protected_qids)
].copy()

# Same person may occur in base + supplement. Prefer the original audited pool.
cand = (
    cand.sort_values(
        ["axis", "_source_priority", "wikidata_id", "norm_name_e2"],
        kind="stable"
    )
    .drop_duplicates("wikidata_id")
    .drop_duplicates(["axis", "norm_name_e2"])
    .reset_index(drop=True)
)

cand["selection_key_e2"] = cand.apply(
    lambda r: selection_key(
        str(r["axis"]), str(r["wikidata_id"]), r["name"]
    ),
    axis=1
)

availability = cand["axis"].value_counts().to_dict()

supply = pd.DataFrame([
    {
        "axis": axis,
        "quota": QUOTAS[axis],
        "fresh_exact_DOB_candidates_after_DEV_CONFIRM_E1_exclusion":
            int(availability.get(axis, 0)),
        "sufficient": int(availability.get(axis, 0)) >= QUOTAS[axis],
        "deficit_if_any":
            max(0, QUOTAS[axis] - int(availability.get(axis, 0))),
    }
    for axis in AXES
])
display(supply)

supply_path = OUT / "V5_DEV_EXPANSION_E2_CANDIDATE_SUPPLY.csv"
supply.to_csv(supply_path, index=False)

supply_sufficient = bool(supply["sufficient"].all())
print("E2 candidate supply sufficient?", supply_sufficient)


,axis,quota,fresh_exact_DOB_candidates_after_DEV_CONFIRM_E1_exclusion,sufficient,deficit_if_any
0,COMPETITIVE,40,55,True,0
1,PROJECT,50,795,True,0
2,STATUS,70,264,True,0


E2 candidate supply sufficient? True


## 2. Hard supply gate, then deterministic E2 selection

In [3]:

if not supply_sufficient:
    decision = {
        "version": "V5_DEV_EXPANSION_E2_FREEZE_DECISION_V1",
        "notebook_version": NOTEBOOK_VERSION,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "status": (
            "V5_DEV_EXPANSION_E2_CANDIDATE_SUPPLY_INSUFFICIENT_"
            "NEEDS_OUTCOME_BLIND_SUPPLEMENT"
        ),
        "trigger_gate_status": cdec["status"],
        "membership_frozen": False,
        "candidate_supply_sha256": sha256_file(supply_path),
        "selection_seed": SEED,
        "axis_quotas_unchanged": QUOTAS,
        "rules": {
            "prior_identity_reused": False,
            "event_outcomes_used": False,
            "pairability_used": False,
            "pair_gap_used": False,
            "chronology_used": False,
            "astrology_used": False,
            "control_used": False,
            "quota_reduction_allowed": False,
        },
        "next_rule": (
            "Stop. Freeze an outcome-blind candidate-source supplement protocol "
            "for only the identity-supply deficit. Do not lower quotas."
        )
    }
    decision_path = OUT / "V5_DEV_EXPANSION_E2_FREEZE_DECISION.json"
    json.dump(
        decision,
        open(decision_path, "w", encoding="utf-8"),
        ensure_ascii=False,
        indent=2
    )
    print(json.dumps(decision, ensure_ascii=False, indent=2))
    raise RuntimeError(
        "E2 exact-DOB candidate supply insufficient. "
        "Do not lower quotas. Send the decision + candidate supply CSV."
    )

chosen_parts = []
for axis in AXES:
    x = (
        cand[cand["axis"] == axis]
        .sort_values(
            ["selection_key_e2", "wikidata_id", "norm_name_e2"]
        )
        .head(QUOTAS[axis])
        .copy()
    )
    assert len(x) == QUOTAS[axis]
    chosen_parts.append(x)

chosen = pd.concat(chosen_parts, ignore_index=True)

assert len(chosen) == 160
assert chosen["wikidata_id"].nunique() == 160
assert chosen["norm_name_e2"].nunique() == 160
assert set(chosen["wikidata_id"].astype(str)).isdisjoint(protected_qids)
assert set(chosen["norm_name_e2"]).isdisjoint(protected_names)

print("E2 deterministic membership selection PASS")


E2 deterministic membership selection PASS


## 3. Assign E2 IDs and the same 8-batch schedule

In [4]:

batch_specs = {
    1: {"COMPETITIVE": 5, "PROJECT": 7, "STATUS": 9},
    2: {"COMPETITIVE": 5, "PROJECT": 7, "STATUS": 9},
    3: {"COMPETITIVE": 5, "PROJECT": 6, "STATUS": 9},
    4: {"COMPETITIVE": 5, "PROJECT": 6, "STATUS": 9},
    5: {"COMPETITIVE": 5, "PROJECT": 6, "STATUS": 9},
    6: {"COMPETITIVE": 5, "PROJECT": 6, "STATUS": 9},
    7: {"COMPETITIVE": 5, "PROJECT": 6, "STATUS": 8},
    8: {"COMPETITIVE": 5, "PROJECT": 6, "STATUS": 8},
}

assert sum(v["COMPETITIVE"] for v in batch_specs.values()) == 40
assert sum(v["PROJECT"] for v in batch_specs.values()) == 50
assert sum(v["STATUS"] for v in batch_specs.values()) == 70

queues = {
    axis: chosen[chosen["axis"] == axis].sort_values(
        ["selection_key_e2", "wikidata_id"]
    ).reset_index(drop=True)
    for axis in AXES
}
ptr = {k: 0 for k in queues}

schedule = []
seq = 1

for b in range(1, 9):
    order = 1
    remaining = batch_specs[b].copy()

    while sum(remaining.values()) > 0:
        for axis in AXES:
            if remaining[axis] <= 0:
                continue

            r = queues[axis].iloc[ptr[axis]]
            ptr[axis] += 1
            remaining[axis] -= 1

            rec = r.to_dict()
            rec.update({
                "subject_id": f"V5DEVE2_{seq:03d}",
                "split": "DEV_EXPANSION_E2",
                "preassigned_axis": axis,
                "expansion_wave": "E2",
                "batch_id": b,
                "research_order_in_batch": order,
                "event_collection_started": False,
                "astrology_scored": False,
            })
            schedule.append(rec)
            seq += 1
            order += 1

roster = pd.DataFrame(schedule)

assert len(roster) == 160
assert roster["subject_id"].nunique() == 160
assert roster["preassigned_axis"].value_counts().to_dict() == {
    "STATUS": 70, "PROJECT": 50, "COMPETITIVE": 40
}

roster = roster.drop(columns=["norm_name_e2"], errors="ignore")

roster_path = OUT / "V5_DEV_EXPANSION_E2_ROSTER_160.csv"
roster.to_csv(roster_path, index=False)

work_cols = [
    "batch_id", "research_order_in_batch", "subject_id", "name",
    "preassigned_axis", "candidate_source", "_candidate_source_e2",
    "birth_date", "birth_place", "gender", "wikidata_id",
    "rodden_rating", "source_row_key"
]
work_cols = [c for c in work_cols if c in roster.columns]
work = roster[work_cols].copy()

work_path = OUT / "V5_DEV_EXPANSION_E2_EVENT_WORKLIST_8BATCH.csv"
work.to_csv(work_path, index=False)

for b in range(1, 9):
    wb = (
        work[work["batch_id"] == b]
        .sort_values("research_order_in_batch")
        .copy()
    )
    wb.to_csv(
        BATCH_OUT / f"V5_DEV_EXPANSION_E2_BATCH_{b:02d}_SUBJECTS.csv",
        index=False
    )

print("E2 roster frozen:", len(roster))
display(
    work.groupby(
        ["batch_id", "preassigned_axis"]
    ).size().unstack(fill_value=0)
)


E2 roster frozen: 160


preassigned_axis,COMPETITIVE,PROJECT,STATUS
batch_id,,,
1,5,7,9
2,5,7,9
3,5,6,9
4,5,6,9
5,5,6,9
6,5,6,9
7,5,6,8
8,5,6,8


## 4. Freeze decision

In [5]:

decision = {
    "version": "V5_DEV_EXPANSION_E2_FREEZE_DECISION_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": "V5_DEV_EXPANSION_E2_ROSTER_FROZEN_READY_FOR_EVENT_COLLECTION",
    "trigger_gate_status": cdec["status"],
    "n_subjects": 160,
    "axis_counts": QUOTAS,
    "roster_sha256": sha256_file(roster_path),
    "worklist_sha256": sha256_file(work_path),
    "candidate_supply_sha256": sha256_file(supply_path),
    "combined_prepair_decision_sha256": sha256_file(COMBINED_DEC),
    "E1_roster_sha256": sha256_file(E1_ROSTER),
    "E1_freeze_decision_sha256": sha256_file(E1_DEC),
    "E2_protocol_sha256": sha256_file(PROTO),
    "selection_seed": SEED,
    "rules": {
        "exact_DOB_candidates_only": True,
        "repaired_DEV_identity_reused": False,
        "sealed_CONFIRM_identity_reused": False,
        "E1_identity_reused": False,
        "event_outcomes_used_for_membership": False,
        "pairability_used_for_membership": False,
        "pair_gap_used_for_membership": False,
        "chronology_used_for_membership": False,
        "astrology_used_for_membership": False,
        "control_used_for_membership": False,
        "project_oversampled_after_observed_deficit": False,
        "axis_quotas_changed": False,
        "confirm_remains_sealed": True
    },
    "event_collection_may_begin": True,
    "pair_generation_allowed": False,
    "astrology_generation_allowed": False,
    "next_rule": (
        "Collect E2 events under the same frozen bounded source-sweep axis contract. "
        "Do not inspect 1-5y pair gaps until E2 event corpus is independently frozen."
    )
}

decision_path = OUT / "V5_DEV_EXPANSION_E2_FREEZE_DECISION.json"
json.dump(
    decision,
    open(decision_path, "w", encoding="utf-8"),
    ensure_ascii=False,
    indent=2
)

print(json.dumps(decision, ensure_ascii=False, indent=2))


{
  "version": "V5_DEV_EXPANSION_E2_FREEZE_DECISION_V1",
  "notebook_version": "SAJU_ML_V5_DEV_EXPANSION_E2_ROSTER_FREEZE_20260817",
  "created_at": "2026-08-17T06:19:50",
  "status": "V5_DEV_EXPANSION_E2_ROSTER_FROZEN_READY_FOR_EVENT_COLLECTION",
  "trigger_gate_status": "V5_COMBINED_PREPAIR_FEASIBILITY_FAIL_EXPANSION_E2_REQUIRED",
  "n_subjects": 160,
  "axis_counts": {
    "COMPETITIVE": 40,
    "PROJECT": 50,
    "STATUS": 70
  },
  "roster_sha256": "0a6be64984c7f7c32037100b45ac41296f81616f4eecf42e49f4e1de3bf69d4c",
  "worklist_sha256": "c3cd8c091d33d285b67c1ec7d8b1894cc4644de0547b490f0134a9c3a3246af9",
  "candidate_supply_sha256": "8c414fb225ca61d05f3b1b0e6dd6a4c72c08b05114d8d5545b1c4c64e733d20b",
  "combined_prepair_decision_sha256": "3d616d89df93906e88bc03c5ca9c814566fa08c48f5c2d13f3b3a392b40c8351",
  "E1_roster_sha256": "76d262c64cc7a83cf6845e96627b8d3cab9be4278fec2c6f07572cc3d1b8e39e",
  "E1_freeze_decision_sha256": "8b21ad8450a9c032a189fe68ffe9804baf7316113844fc399a284bbc223e


## Send back

### If the notebook **passes**

Send:

```text
V5_DEV_EXPANSION_E2_FREEZE_DECISION.json
V5_DEV_EXPANSION_E2_CANDIDATE_SUPPLY.csv
V5_DEV_EXPANSION_E2_EVENT_WORKLIST_8BATCH.csv
```

You do **not** need to upload all eight batch files separately; the combined worklist
contains the same 160 frozen subjects and batch assignments.

### If it stops for candidate-supply insufficiency

Send only:

```text
V5_DEV_EXPANSION_E2_FREEZE_DECISION.json
V5_DEV_EXPANSION_E2_CANDIDATE_SUPPLY.csv
```

Do not lower quotas or manually substitute people.
